# GBM予測 & 3銘柄ポートフォリオ

対話的UI:
1. **1銘柄GBM予測** — μ・σ調整可、±1〜3σシグマバンド、VaR 95%/99%
2. **3銘柄ポートフォリオ** — 相関ベースの真のリスク、分散効果、ポートフォリオVaR

### 推定モードのデフォルト
**過去2年の60日（3ヶ月）ローリングμ・σの平均**（= `2年平均` モード）

メインのGBMチャート下に **過去2年の60日ローリングμ・σ水位** を表示。今使ってる値（赤破線）が水位のどこにいるか確認しながら調整できる。

### 銘柄3は任意銘柄OK
ルール: 時価総額100億円以上 + 上場1年以上を満たせば任意。
銘柄3には **ティッカーを直接入力** 可能（例: `6098.T`、`8035.T`）。指定30銘柄名でも認識。

## 1. セットアップ & データ取得

In [12]:
import logging, warnings
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from scipy.stats import norm
import ipywidgets as widgets
from IPython.display import display

plt.rcParams['font.family'] = 'Hiragino Sans'
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', context='notebook', font='Hiragino Sans', rc={'axes.unicode_minus': False})

ROOT = Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd()

TRADING_DAYS = 252
RF = 0.0
ROLL_WINDOW = 60
HISTORY_YEARS = 2

to_date = datetime.today()
from_date = to_date - timedelta(days=365 * (HISTORY_YEARS + 1))
FROM_STR = from_date.strftime('%Y-%m-%d')
TO_STR = to_date.strftime('%Y-%m-%d')
HISTORY_CUTOFF = pd.Timestamp(to_date) - pd.Timedelta(days=365 * HISTORY_YEARS)
print(f'取得期間: {from_date.date()} 〜 {to_date.date()}')

取得期間: 2023-05-20 〜 2026-05-19


In [13]:
stocks = pd.read_csv(ROOT / 'data' / 'stocks.csv')
stocks['ticker'] = stocks['code'].astype(str).str.zfill(4) + '.T'
ticker_to_name = dict(zip(stocks['ticker'], stocks['name']))

raw = yf.download(
    stocks['ticker'].tolist(),
    start=FROM_STR, end=TO_STR,
    auto_adjust=True, progress=False, group_by='ticker',
)
close = pd.concat(
    {ticker_to_name[t]: raw[t]['Close'] for t in stocks['ticker'] if t in raw.columns.get_level_values(0)},
    axis=1,
).dropna(how='all').sort_index()
returns = close.pct_change().dropna(how='all')
ALL_STOCKS = list(returns.columns)
print(f'指定30銘柄: 価格 {close.shape}, リターン {returns.shape}')

指定30銘柄: 価格 (730, 30), リターン (729, 30)


## 2. GBM / VaR / 水位 / 任意ティッカー解決ユーティリティ

In [14]:
def gbm_bands(S0, mu, sigma, horizon_days):
    dt = 1 / TRADING_DAYS
    t = np.arange(horizon_days + 1) * dt
    log_mean = (mu - sigma**2 / 2) * t
    log_std = sigma * np.sqrt(t)
    median = S0 * np.exp(log_mean)
    expected = S0 * np.exp(mu * t)
    bands = {k: (S0 * np.exp(log_mean - k * log_std), S0 * np.exp(log_mean + k * log_std)) for k in [1, 2, 3]}
    return t, median, expected, bands

def var_levels(S0, mu, sigma, horizon_days, confidences=(0.95, 0.99)):
    T = horizon_days / TRADING_DAYS
    log_mean_T = (mu - sigma**2 / 2) * T
    log_std_T = sigma * np.sqrt(T)
    out = {}
    for c in confidences:
        price = S0 * np.exp(log_mean_T + log_std_T * norm.ppf(1 - c))
        out[c] = {'price': price, 'loss_pct': 1 - price / S0, 'loss_jpy': S0 - price}
    return out

def rolling_mu_sigma(ret_series, window=ROLL_WINDOW):
    mu = ret_series.rolling(window).mean() * TRADING_DAYS
    sigma = ret_series.rolling(window).std() * np.sqrt(TRADING_DAYS)
    return mu, sigma

WINDOW_OPTIONS = ['2年平均', '直近10日', '直近20日', '直近60日', '直近120日', '直近252日']

def estimate_mu_sigma_from_ret(r, mode):
    if mode == '2年平均':
        mu_roll, sigma_roll = rolling_mu_sigma(r)
        mu_roll = mu_roll[mu_roll.index >= HISTORY_CUTOFF]
        sigma_roll = sigma_roll[sigma_roll.index >= HISTORY_CUTOFF]
        return mu_roll.mean(), sigma_roll.mean()
    window = int(mode.replace('直近', '').replace('日', ''))
    recent = r.iloc[-window:]
    return recent.mean() * TRADING_DAYS, recent.std() * np.sqrt(TRADING_DAYS)

extra_cache = {}

def resolve_stock(value):
    value = value.strip()
    if value in ALL_STOCKS:
        return value, close[value].dropna(), returns[value].dropna()
    ticker = value.upper()
    if ticker.isdigit():
        ticker = ticker.zfill(4) + '.T'
    elif not ticker.endswith('.T'):
        if ticker.replace('.', '').isdigit():
            ticker = ticker.split('.')[0].zfill(4) + '.T'
        else:
            ticker = ticker + '.T'
    if ticker in extra_cache:
        return extra_cache[ticker]
    try:
        df = yf.download(ticker, start=FROM_STR, end=TO_STR, auto_adjust=True, progress=False)
        if df.empty:
            return None
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        c = df['Close'].dropna() if 'Close' in df.columns else df.iloc[:, 0].dropna()
        if isinstance(c, pd.DataFrame):
            c = c.iloc[:, 0]
        r = c.pct_change().dropna()
        result = (ticker, c, r)
        extra_cache[ticker] = result
        return result
    except Exception:
        return None

BAND_PALETTE = ['#fde68a', '#fdba74', '#fca5a5']

---

## 3. 1銘柄 GBM予測（対話UI）

**操作**: 銘柄/推定モード/予測日数を切替で即時再描画。`μ/σ手動上書き` ON で下段の水位を見ながらスライダー調整。

In [ ]:
def plot_single_forecast(stock, mode, horizon_days, mu_override, sigma_override, use_override):
    series = close[stock].dropna()
    S0 = series.iloc[-1]
    last_date = series.index[-1]
    r = returns[stock].dropna()

    mu_est, sigma_est = estimate_mu_sigma_from_ret(r, mode)
    mu = mu_override if use_override else mu_est
    sigma = sigma_override if use_override else sigma_est

    _, median, expected, bands = gbm_bands(S0, mu, sigma, horizon_days)
    var = var_levels(S0, mu, sigma, horizon_days)
    forecast_dates = pd.bdate_range(last_date, periods=horizon_days + 1)
    hist = series.iloc[-90:]

    mu_roll, sigma_roll = rolling_mu_sigma(r)
    mu_roll = mu_roll[mu_roll.index >= HISTORY_CUTOFF]
    sigma_roll = sigma_roll[sigma_roll.index >= HISTORY_CUTOFF]
    mu_2y_avg = mu_roll.mean()
    sigma_2y_avg = sigma_roll.mean()

    fig = plt.figure(figsize=(14, 11))
    gs = fig.add_gridspec(3, 1, height_ratios=[3, 1, 1], hspace=0.5)

    ax = fig.add_subplot(gs[0, 0])
    ax.plot(hist.index, hist.values, color='#0f172a', linewidth=1.8, label='実績')
    for (k, (lo, hi)), c in zip(bands.items(), BAND_PALETTE):
        ax.fill_between(forecast_dates, lo, hi, color=c, alpha=0.6, label=f'±{k}σ')
    ax.plot(forecast_dates, median, color='#1e293b', linestyle='--', linewidth=1.6, label='中央値')
    ax.plot(forecast_dates, expected, color='#2563eb', linestyle=':', linewidth=1.4, label='期待値')
    ax.axhline(var[0.95]['price'], color='#dc2626', linewidth=1.3, linestyle='-.', alpha=0.85)
    ax.axhline(var[0.99]['price'], color='#7f1d1d', linewidth=1.3, linestyle='-.', alpha=0.85)
    ax.annotate(f"VaR 95% : ¥{var[0.95]['price']:,.0f} ({var[0.95]['loss_pct']:.1%})",
                xy=(forecast_dates[-1], var[0.95]['price']), xytext=(8, 0), textcoords='offset points',
                color='#dc2626', va='center', fontsize=9, fontweight='bold')
    ax.annotate(f"VaR 99% : ¥{var[0.99]['price']:,.0f} ({var[0.99]['loss_pct']:.1%})",
                xy=(forecast_dates[-1], var[0.99]['price']), xytext=(8, 0), textcoords='offset points',
                color='#7f1d1d', va='center', fontsize=9, fontweight='bold')
    ax.axvline(last_date, color='gray', linewidth=0.6)
    used = '手動' if use_override else mode
    ax.set_title(
        f'{stock} — GBM予測（{horizon_days}営業日 ≒ {horizon_days/21:.1f}ヶ月）\n'
        f'現在値 ¥{S0:,.0f} / μ={mu:.1%} / σ={sigma:.1%} ({used})',
        fontsize=12, fontweight='bold'
    )
    ax.set_ylabel('株価 (円)')
    ax.legend(loc='upper left', fontsize=9, frameon=True)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'¥{x:,.0f}'))

    ax_mu = fig.add_subplot(gs[1, 0])
    ax_mu.plot(mu_roll.index, mu_roll.values, color='#0ea5e9', linewidth=1.6)
    ax_mu.fill_between(mu_roll.index, mu_roll.values, 0, color='#0ea5e9', alpha=0.15)
    ax_mu.axhline(0, color='gray', linewidth=0.5)
    ax_mu.axhline(mu_2y_avg, color='#0ea5e9', linewidth=1.0, linestyle=':', label=f'2年平均 {mu_2y_avg:.1%}')
    ax_mu.axhline(mu, color='#dc2626', linewidth=1.5, linestyle='--', label=f'今使ってる値 {mu:.1%}')
    ax_mu.set_title('μ（年率）過去2年水位 — 60日ローリング', fontsize=10, fontweight='bold')
    ax_mu.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax_mu.legend(loc='upper left', fontsize=8)

    ax_sigma = fig.add_subplot(gs[2, 0])
    ax_sigma.plot(sigma_roll.index, sigma_roll.values, color='#dc2626', linewidth=1.6)
    ax_sigma.fill_between(sigma_roll.index, sigma_roll.values, 0, color='#dc2626', alpha=0.15)
    ax_sigma.axhline(sigma_2y_avg, color='#dc2626', linewidth=1.0, linestyle=':', label=f'2年平均 {sigma_2y_avg:.1%}')
    ax_sigma.axhline(sigma, color='#0f172a', linewidth=1.5, linestyle='--', label=f'今使ってる値 {sigma:.1%}')
    ax_sigma.set_title('σ（年率）過去2年水位 — 60日ローリング', fontsize=10, fontweight='bold')
    ax_sigma.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax_sigma.legend(loc='upper left', fontsize=8)

    plt.show()

stock_dd = widgets.Dropdown(options=ALL_STOCKS, value='任天堂', description='銘柄')
mode_dd = widgets.Dropdown(options=WINDOW_OPTIONS, value='2年平均', description='推定モード')
horizon_sl = widgets.IntSlider(value=63, min=21, max=126, step=7, description='予測日数', continuous_update=False)
use_override_cb = widgets.Checkbox(value=False, description='μ/σ手動上書き')
mu_sl = widgets.FloatSlider(value=0.10, min=-1.0, max=1.5, step=0.05, description='μ(年率)', readout_format='.0%', continuous_update=False)
sigma_sl = widgets.FloatSlider(value=0.30, min=0.05, max=1.5, step=0.05, description='σ(年率)', readout_format='.0%', continuous_update=False)

ui = widgets.VBox([
    widgets.HBox([stock_dd, mode_dd, horizon_sl]),
    widgets.HBox([use_override_cb, mu_sl, sigma_sl]),
])
out = widgets.interactive_output(plot_single_forecast, {
    'stock': stock_dd, 'mode': mode_dd, 'horizon_days': horizon_sl,
    'mu_override': mu_sl, 'sigma_override': sigma_sl, 'use_override': use_override_cb,
})
display(ui, out)

Output()

---

## 4. 3銘柄ポートフォリオ（対話UI）

- 銘柄1・銘柄2: 指定30銘柄から選択
- **銘柄3: 任意銘柄OK**（ティッカー直接入力可、例: `6098.T`、`8035.T`、`9433.T`）
- ウェイトは自動正規化、相関行列ベースで真の $\sigma_p$ 算出
- 過去2年の $\mu_p / \sigma_p$ 水位グラフで今使ってる値が極端でないかチェック

In [5]:
def plot_portfolio(s1, s2, s3, w1, w2, w3, mode, horizon_days):
    res = resolve_stock(s3)
    if res is None:
        print(f"❌ 銘柄3 '{s3}' を取得できませんでした。ティッカー（例: 6098.T）または指定銘柄名を入力してください。")
        return
    s3_name, s3_close, s3_ret = res

    raw_w = np.array([w1, w2, w3], dtype=float)
    if raw_w.sum() == 0:
        raw_w = np.ones(3)
    w = raw_w / raw_w.sum()

    ret_df = pd.concat({s1: returns[s1], s2: returns[s2], s3_name: s3_ret}, axis=1).dropna()
    names = [s1, s2, s3_name]

    if mode == '2年平均':
        r_used = ret_df[ret_df.index >= HISTORY_CUTOFF]
    else:
        window = int(mode.replace('直近', '').replace('日', ''))
        r_used = ret_df.iloc[-window:]

    mu_i = r_used.mean().values * TRADING_DAYS
    cov = r_used.cov().values * TRADING_DAYS
    corr = r_used.corr().values
    sigma_i = np.sqrt(np.diag(cov))
    mu_p = float(w @ mu_i)
    sigma_p = float(np.sqrt(w @ cov @ w))
    sigma_naive = float(w @ sigma_i)
    diversification = 1 - sigma_p / sigma_naive if sigma_naive > 0 else 0
    sharpe_p = (mu_p - RF) / sigma_p if sigma_p > 0 else np.nan

    S0 = 100.0
    _, median, expected, bands = gbm_bands(S0, mu_p, sigma_p, horizon_days)
    var = var_levels(S0, mu_p, sigma_p, horizon_days)
    last_date = ret_df.index[-1]
    forecast_dates = pd.bdate_range(last_date, periods=horizon_days + 1)

    hist_window = 90
    port_ret = (ret_df.iloc[-hist_window:] * w).sum(axis=1)
    hist_value = (1 + port_ret).cumprod()
    hist_value = hist_value / hist_value.iloc[-1] * S0

    port_ret_all = (ret_df * w).sum(axis=1).dropna()
    mu_p_roll, sigma_p_roll = rolling_mu_sigma(port_ret_all)
    mu_p_roll = mu_p_roll[mu_p_roll.index >= HISTORY_CUTOFF]
    sigma_p_roll = sigma_p_roll[sigma_p_roll.index >= HISTORY_CUTOFF]
    mu_p_2y = mu_p_roll.mean()
    sigma_p_2y = sigma_p_roll.mean()

    fig = plt.figure(figsize=(16, 16))
    gs = fig.add_gridspec(4, 3, height_ratios=[3, 2, 1, 1], hspace=0.55, wspace=0.35)

    ax_main = fig.add_subplot(gs[0, :])
    ax_main.plot(hist_value.index, hist_value.values, color='#0f172a', linewidth=1.8, label='実績(正規化)')
    for (k, (lo, hi)), c in zip(bands.items(), BAND_PALETTE):
        ax_main.fill_between(forecast_dates, lo, hi, color=c, alpha=0.6, label=f'±{k}σ')
    ax_main.plot(forecast_dates, median, color='#1e293b', linestyle='--', linewidth=1.6, label='中央値')
    ax_main.plot(forecast_dates, expected, color='#2563eb', linestyle=':', linewidth=1.4, label='期待値')
    ax_main.axhline(var[0.95]['price'], color='#dc2626', linewidth=1.3, linestyle='-.', alpha=0.85)
    ax_main.axhline(var[0.99]['price'], color='#7f1d1d', linewidth=1.3, linestyle='-.', alpha=0.85)
    ax_main.annotate(f"VaR 95% : {var[0.95]['price']:.1f} ({var[0.95]['loss_pct']:.1%})",
                     xy=(forecast_dates[-1], var[0.95]['price']), xytext=(8, 0), textcoords='offset points',
                     color='#dc2626', va='center', fontsize=9, fontweight='bold')
    ax_main.annotate(f"VaR 99% : {var[0.99]['price']:.1f} ({var[0.99]['loss_pct']:.1%})",
                     xy=(forecast_dates[-1], var[0.99]['price']), xytext=(8, 0), textcoords='offset points',
                     color='#7f1d1d', va='center', fontsize=9, fontweight='bold')
    ax_main.axvline(last_date, color='gray', linewidth=0.6)
    ax_main.set_title(
        f'ポートフォリオ予測 — {horizon_days}営業日（≒{horizon_days/21:.1f}ヶ月）  ({mode})\n'
        f'μ_p={mu_p:.1%}  σ_p={sigma_p:.1%}  Sharpe={sharpe_p:.2f}  分散効果={diversification:.1%}',
        fontsize=12, fontweight='bold'
    )
    ax_main.legend(loc='upper left', fontsize=9, frameon=True)
    ax_main.set_ylabel('ポートフォリオ価値（初期=100）')

    palette3 = ['#ef4444', '#3b82f6', '#10b981']
    short_names = [n[:10] for n in names]

    ax_w = fig.add_subplot(gs[1, 0])
    bars = ax_w.barh(short_names, w * 100, color=palette3)
    for bar, val in zip(bars, w * 100):
        ax_w.text(val + 1, bar.get_y() + bar.get_height()/2, f'{val:.0f}%', va='center', fontsize=9)
    ax_w.set_xlim(0, max(w * 100) * 1.3 + 5)
    ax_w.set_title('ウェイト', fontweight='bold')
    ax_w.invert_yaxis()

    ax_metrics = fig.add_subplot(gs[1, 1])
    x = np.arange(3)
    ax_metrics.bar(x - 0.2, mu_i * 100, width=0.4, color='#0ea5e9', label='μ(年率)')
    ax_metrics.bar(x + 0.2, sigma_i * 100, width=0.4, color='#dc2626', label='σ(年率)')
    ax_metrics.set_xticks(x)
    ax_metrics.set_xticklabels(short_names, fontsize=8, rotation=15)
    ax_metrics.set_ylabel('%')
    ax_metrics.legend(fontsize=8)
    ax_metrics.set_title('個別 μ / σ', fontweight='bold')
    ax_metrics.axhline(0, color='gray', linewidth=0.5)

    ax_corr = fig.add_subplot(gs[1, 2])
    sns.heatmap(
        pd.DataFrame(corr, index=short_names, columns=short_names),
        annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, center=0,
        ax=ax_corr, cbar_kws={'shrink': 0.7}, square=False, annot_kws={'fontsize': 10}
    )
    ax_corr.set_title('相関行列', fontweight='bold')

    ax_mu = fig.add_subplot(gs[2, :])
    ax_mu.plot(mu_p_roll.index, mu_p_roll.values, color='#0ea5e9', linewidth=1.6)
    ax_mu.fill_between(mu_p_roll.index, mu_p_roll.values, 0, color='#0ea5e9', alpha=0.15)
    ax_mu.axhline(0, color='gray', linewidth=0.5)
    ax_mu.axhline(mu_p_2y, color='#0ea5e9', linewidth=1.0, linestyle=':', label=f'2年平均 {mu_p_2y:.1%}')
    ax_mu.axhline(mu_p, color='#dc2626', linewidth=1.5, linestyle='--', label=f'今使ってる値 {mu_p:.1%}')
    ax_mu.set_title('μ_p（年率）過去2年水位 — 同ウェイトでの60日ローリング', fontsize=10, fontweight='bold')
    ax_mu.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax_mu.legend(loc='upper left', fontsize=8)

    ax_sigma = fig.add_subplot(gs[3, :])
    ax_sigma.plot(sigma_p_roll.index, sigma_p_roll.values, color='#dc2626', linewidth=1.6)
    ax_sigma.fill_between(sigma_p_roll.index, sigma_p_roll.values, 0, color='#dc2626', alpha=0.15)
    ax_sigma.axhline(sigma_p_2y, color='#dc2626', linewidth=1.0, linestyle=':', label=f'2年平均 {sigma_p_2y:.1%}')
    ax_sigma.axhline(sigma_p, color='#0f172a', linewidth=1.5, linestyle='--', label=f'今使ってる値 {sigma_p:.1%}')
    ax_sigma.set_title('σ_p（年率）過去2年水位 — 同ウェイトでの60日ローリング', fontsize=10, fontweight='bold')
    ax_sigma.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax_sigma.legend(loc='upper left', fontsize=8)

    plt.show()

ps1 = widgets.Dropdown(options=ALL_STOCKS, value='任天堂', description='銘柄1')
ps2 = widgets.Dropdown(options=ALL_STOCKS, value='ソニーグループ', description='銘柄2')
ps3 = widgets.Text(value='6098.T', description='銘柄3', placeholder='ティッカー(例: 6098.T) or 指定銘柄名')
pw1 = widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05, description='w1', readout_format='.2f', continuous_update=False)
pw2 = widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05, description='w2', readout_format='.2f', continuous_update=False)
pw3 = widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05, description='w3', readout_format='.2f', continuous_update=False)
pmode = widgets.Dropdown(options=WINDOW_OPTIONS, value='2年平均', description='推定モード')
phorizon = widgets.IntSlider(value=63, min=21, max=126, step=7, description='予測日数', continuous_update=False)

ui_p = widgets.VBox([
    widgets.HBox([ps1, pw1]),
    widgets.HBox([ps2, pw2]),
    widgets.HBox([ps3, pw3]),
    widgets.HBox([pmode, phorizon]),
])
out_p = widgets.interactive_output(plot_portfolio, {
    's1': ps1, 's2': ps2, 's3': ps3, 'w1': pw1, 'w2': pw2, 'w3': pw3,
    'mode': pmode, 'horizon_days': phorizon,
})
display(ui_p, out_p)

Output()

### 読み方
- **μ_p**: ポートフォリオ年率期待リターン（個別μの加重和）
- **σ_p**: 共分散行列ベースのポートフォリオ年率ボラ（相関を考慮した真のリスク）
- **分散効果**: $1 - \sigma_p / \sigma_{naive}$。値が大きいほど相関の低い銘柄を組合せている
- **VaR 95% / 99%**: 3ヶ月後の最悪損失水準
- **相関行列**: 0.5以上は同方向 → 分散効果が薄い。負相関はヘッジ効果
- **下段の水位グラフ**: いま使ってる値（赤破線）が過去の振れ幅のどこにいるかチェック。極端なら推定モード or 手動値を見直す

150万円制約に置き換えるなら、`VaR 95%の損失%` × 1.5M円 が「3ヶ月で覚悟する金額」。